Import Necessary Libraries

In [1]:
%matplotlib qt
#The version of the packages used in this notebook are listed in the comments
import numpy as np #1.21.5
import matplotlib.pyplot as plt #3.5.2
from matplotlib.widgets import RangeSlider, Slider 
from PyQt5.QtWidgets import QFileDialog  #5.12.3
import cv2 #4.5.5
from skimage import exposure #0.19.2
import sys

Define Functions

In [2]:
def update_progress(progress):
    """Print a progress bar"""
    barLength = 10 # Modify this Xto change the length of the progress bar
    status = ""
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
        status = "error: progress var must be float\r\n"
    if progress < 0:
        progress = 0
        status = "Halt...\r\n"
    if progress >= 1:
        progress = 1
        status = "Done...\r\n"
    block = int(round(barLength*progress))
    text = "\rPercent: [{0}] {1}% {2}".format( "#"*block + "-"*(barLength-block), progress*100, status)
    sys.stdout.write(text)
    sys.stdout.flush()

def noframeimshow(im,contrast=None, greyscale=True, size=5):
    """To display an image without the frame, will pause the program until the figure is closed
    im: (n,m) image to display
    contrast: [min,max] minimum and maximum value to display 
    greyscale: True if the image is greyscale, otherwise use matplotlib default colormap
    size: size of the figure in inches
    """
    if contrast is None:                     #if no contrast is given, use the min and max of the image
        contrast=[np.min(im),np.max(im)]
    y, x = np.shape(im)                      #get the size of the image
    fig = plt.figure(figsize=(size,size*y/x))       #create a figure with the right size
    fig.set_size_inches(size,size*y/x, forward=False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.axis(False)
    fig.add_axes(ax)
    if greyscale:
        ax.imshow(im,vmin=contrast[0], vmax=contrast[1], cmap='gray')
    else:
        ax.imshow(im,vmin=contrast[0], vmax=contrast[1])
    plt.show()
    fnum=fig.number
    while True:       #wait until the figure is closed        
        plt.pause(0.1)
        if plt.fignum_exists(fnum):
            continue
        else:
            break
    pass


def plot_slider(im, title=None, return_image=True):
    """To display an image with a slider to change the contrast and gamma, will pause the program until the figure is closed
    im: (n,m) image to display
    title: title of the figure
    return_image: True if the function should return the image after the contrast and gamma are changed
    
    The gamma slider use the skimage.exposure.adjust_gamma function to adjust the gamma of the image
    The contrast slider set the color limits in the matplotlib.pyplot.imshow function to adjust the contrast of the image

    return:
    im_gamma: (n,m) image after gamma are changed
    contrast: [min,max] minimum and maximum value to display"""
    global gamma_slider, contrast_slider, fig, im_gamma, vmin, vmax
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111)
    ax.set_title(title)
    p=plt.imshow(im)
    ax.set_aspect('equal')
    plt.subplots_adjust(bottom=.25)
    ax_gslide=plt.axes([.2,.1,.65,.03])
    ax_cslide=plt.axes([.2,.15,.65,.03])
    gamma_slider=Slider(ax_gslide,"Gamma",valmin=-10,valmax=10,valinit=0,valstep=.05)
    contrast_slider=RangeSlider(ax_cslide,"Contrast",valmin=0,valmax=1,valinit=(0,1),valstep=.05)
    M=np.max(im)
    im_gamma=im.copy()
    vmin=np.min(im)
    vmax=np.max(im)
    def update_gamma(val):
        global im_gamma
        gamma=np.exp(gamma_slider.val)
        im_gamma=exposure.adjust_gamma(im, gamma)
        p.set_data(im_gamma)
        fig.canvas.draw()
    def update_contrast(val):
        global vmin, vmax
        vmin=val[0]*M
        vmax=val[1]*M
        p.set_clim(vmin, vmax)
        fig.canvas.draw()           
    gamma_slider.on_changed(update_gamma)
    contrast_slider.on_changed(update_contrast)
    plt.show()
    fnum=fig.number
    while True:
        plt.pause(0.1)
        if plt.fignum_exists(fnum):
            continue
        else:
            break
    contrast=(vmin,vmax)
    if return_image:
        return im_gamma, contrast

def logimage(im):
    """Logarithmic transform an image
    im: (n,m) image to transform

    return
    im: transformed image
    """
    im=im-np.min(im)   #First linear transform the image to [1,10]
    im=9*im/np.max(im)+1   
    im=np.log10(im)   #Take the log10 of the image, will be in [0,1]
    return im


def virtual_dark_field(data, vapt):
    """Calculate the virtual dark field of a dataset, an append method is used to save memory during calculation
    data: (n, m, kn, km) 4D dataset
    vapt: (kn, km) virtual aperture used to calculate the virtual dark field. Should be the same size as the diffraction pattern. Values should only be 0 or 1, where 1 is the region of interest

    return
    vdf: (n, m) virtual dark field image of the dataset
    """
    n=len(data[0])
    vdf= np.empty((0,n))
    for i, part in enumerate(data):
        vdf=np.append(vdf,[np.sum(part[:]*vapt, axis=(1,2))], axis=0)
    return vdf

def SADP(data, roi):
    """Calculate the mean of all diffraction patterns in a region of interest
    data: (n, m, kn, km) 4D dataset
    roi: (n, m) region of interest. Values should only be 0 or 1, where 1 is the region of interest
    
    return
    sadp: (kn, km) mean diffraction pattern of the region of interest"""
    sadp = np.mean(data[roi==1],axis=0)
    return sadp

def FTEM(data):
    """Calculate the normalized variance map of the diffraction patterns in a dataset
    data: (n, m, kn, km) 4D dataset
    
    return
    V: (kn, km) normalized variance map of the dataset"""
    m = len(data[0,0])
    V = np.zeros((m,m)) #initialize the normalized variance map
    for i in range(m):
        for j in range(m):
            v = data[:, :, i, j].astype(float).flatten() #List all the diffraction signals at the pixel (i,j)
            V[i,j] = (np.mean(v**2)/(np.mean(v)**2)) - 1 #Calculate the normalized variance of all the diffraction signals at the pixel (i,j)
        if i%10 == 0:
            update_progress((i+1)/m)
    return V


def corr_coef_map(data, ref_k_roi):
    """Calculate the correlation coefficient map of a dataset with a reference k vector
    data: (n, m, kn, km) 4D dataset, should be the diffuse scattering dataset
    ref_k_roi: (kn, km) region of interest of the reference k vector. Values should only be 0 or 1, where 1 is the region of interest
                ref_k_roi should be the same size as the diffraction pattern
                ref_k_roi can have more than one pixel of roi, then the averaged flucuation of all roi pixels will be used as the reference

    return
    corrmap: (n, m) correlation coefficient map of the dataset with the reference k vector
    """
    refvdf = virtual_dark_field(data, ref_k_roi) #calculate the virtual dark field of the reference k vector
    corrmap = np.zeros_like(data[0,0]).astype(float) #initialize the correlation coefficient map
    for i in range(len(corrmap)):
        for j in range(len(corrmap)):
            targetvdf = data[:,:,i,j] #calculate the virtual dark field of the target k vector
            corrmap[i,j] = np.corrcoef(refvdf.flatten(), targetvdf.flatten())[0,1] #calculate the correlation coefficient
        if (i+1)%10 == 0:
            update_progress((i+1)/len(corrmap))
    return corrmap

Load preprocessed data and I_diff

In [19]:
"""select the proprocessed dataset, and the I_diff will be loaded automatically
The example dataset has a shape of (200, 200, 150, 150)"""
filename = QFileDialog.getOpenFileName( caption="Open Image", filter="Raw Images (*.tif) ;; Image Files (*.png *.jpg *.bmp) ;; EMI Files (*.emi) ;; Numpy Files (*.npy) ;; Hyperspy Files (*.hspy)", initialFilter="Numpy Files (*.npy)")[0]
filename = filename[:-4]
bindata = np.load(filename+'.npy') #load the preprocessed dataset
print('Preprocessed dataset loaded, shape: ', np.shape(bindata))
I_diff = np.load(filename+'_I_diff.npy') #load I_diff
print('I_diff loaded, shape: ', np.shape(I_diff))

Preprocessed dataset loaded, shape:  (200, 200, 150, 150)
I_diff loaded, shape:  (200, 200, 150, 150)


Calculate the averaged diffraction pattern

In [4]:
avgdp = np.mean(bindata, axis=(0,1)) #calculate the average diffraction pattern
#visualize the average diffraction pattern
avgdplog = logimage(logimage(avgdp)) #use multiple log transformation to enhance the contrast of diffuse scattering
noframeimshow(avgdplog, contrast=[np.min(avgdplog), np.max(avgdplog)*0.2], greyscale=True, size=5)

Calculate Normalize Variance Map

In [5]:
V = FTEM(I_diff) #calculate the normalized variance map using the diffuse scattering dataset
im, contrast = plot_slider(V, title='Normalized Variance Map') #display the normalized variance map with a slider to change the contrast and gamma
noframeimshow(im, contrast=contrast, greyscale=False) #display the normalized variance map without frame for saving
#The normalized variance map is left with defaut colormap, so it will not be confused with the diffraction pattern

Percent: [#########-] 94.0% 33333333333%  

Calculate Correlation Coefficient Map

In [15]:
k_ref = [42, 78] #the index of the reference k vector, the [42,78] is the index for 1/3(422) peak in the example dataset 
roi = np.zeros_like(I_diff[0,0]).astype(int) #initialize the region of interest
roi[k_ref[0], k_ref[1]] = 1 #set the reference k vector as the region of interest
corrmap = corr_coef_map(I_diff, roi) #calculate the correlation coefficient map


Percent: [##########] 100% Done...33333%  


In [14]:
#Visualize the correlation coefficient map as image
corrmap_nn = corrmap.copy() #make a copy of the correlation coefficient map
corrmap_nn[corrmap<0] = 0 #set the negative correlation coefficient to 0 otherwise the gamma correction won't work
im, contrast = plot_slider(corrmap_nn, title='Correlation Coefficient Map') #display the correlation coefficient map with a slider to change the contrast and gamma
noframeimshow(im, contrast=contrast, greyscale=True) #display the correlation coefficient map without frame for saving


In [16]:
#Visualize the correlation coefficient map as contour plot
level1 = [0.02, 0.06, 0.10, 0.14] #set appropriate contour levels to visualize the correlation coefficient map
contrast1 = (0.02, 0.14)
corrmap1 = np.flip(corrmap, axis=0) #flip the correlation coefficient map to fit the way contour plot is displayed
f, ax = plt.subplots(1,1)
# f, ax = plt.subplots(1)
CSF = ax.contourf(corrmap1, levels=level1, extend='both', vmin=contrast1[0], vmax=contrast1[1], cmap='Greys')
CS = ax.contour(corrmap1, levels=level1, colors='k', vmin=contrast1[0], vmax=contrast1[1])
#set the bounding box to square
plt.gca().set_aspect('equal', adjustable='box')
# ax.clabel(CS, CS.levels, inline=True, fontsize=6, manual=True)
# plt.colorbar(CSF)
plt.show()

Calculate Virtual Dark Filed from Diffuse Scattering Patterns

In [10]:
"""Select the indices of pixels of the virtual aperture, where signals will be use to calculate the VDF
The indices given here are for the 1/3{422} peaks in the example dataset"""
vapt_ix = np.array([ 41,  41,  42,  42,  42,  42,  43,  43,  43,  43,  44,  44,  44,
        44,  45,  45,  55,  55,  55,  55,  56,  56,  56,  56,  57,  57,
        57,  57,  58,  58,  58,  58,  59,  59,  59,  60,  60,  60,  60,
        60,  61,  61,  61,  61,  61,  62,  62,  62,  62,  63,  86,  86,
        86,  87,  87,  87,  87,  88,  88,  88,  88,  89,  89,  89,  89,
        90,  90,  90,  90,  91,  91,  91,  91,  91,  92,  92,  92,  92,
        92,  93,  93,  93,  93,  93,  94, 103, 104, 104, 104, 104, 104,
       105, 105, 105, 105, 105, 106, 106, 106, 106, 106, 107, 107, 107])
vapt_iy = np.array([ 77,  78,  76,  77,  78,  79,  76,  77,  78,  79,  76,  77,  78,
        79,  77,  78,  48,  49,  50,  51,  48,  49,  50,  51,  48,  49,
        50,  51,  48,  49,  50,  51, 102, 103, 104, 101, 102, 103, 104,
       105, 101, 102, 103, 104, 105, 102, 103, 104, 105, 103,  46,  47,
        48,  46,  47,  48,  49,  46,  47,  48,  49,  46,  47,  48,  49,
        47, 100, 101, 102,  99, 100, 101, 102, 103,  99, 100, 101, 102,
       103,  99, 100, 101, 102, 103, 101,  73,  71,  72,  73,  74,  75,
        71,  72,  73,  74,  75,  71,  72,  73,  74,  75,  72,  73,  74])

In [11]:
vapt = np.zeros_like(I_diff[0,0]).astype(int) #initialize the virtual aperture
for x, y in zip(vapt_ix, vapt_iy):
    vapt[x,y] = 1 #set the pixels of the virtual aperture to 1

noframeimshow(vapt) #display the virtual aperture to check if it is correct

In [12]:
vdf = virtual_dark_field(I_diff, vapt) #calculate the virtual dark field of the dataset
noframeimshow(vdf) #display the virtual dark field without frame for saving

Obtain Domain Map via Z-score Outlier Detection

In [13]:
Z = (vdf-np.mean(vdf))/np.std(vdf) #calculate the Z-score of each pixel in the virtual dark field
domains = np.zeros_like(Z).astype(int) #initialize the domain map
domains[Z>=3] = 1 #set the pixels with Z-score >= 3 to 1
noframeimshow(domains) #display the domain map

In [14]:
"Check whether the averaged diffraction pattern of all the domains has the diffraction peaks of interest"
avgdp_domain = SADP(bindata, domains) #calculate the averaged diffraction pattern of all the domains
im, contrast = plot_slider(avgdp_domain, title='Averaged Diffraction Pattern of Domains') #display the averaged diffraction pattern of all the domains with a slider to change the contrast and gamma
noframeimshow(im, contrast=contrast, greyscale=True) #display the averaged diffraction pattern of all the domains without frame for saving

In [15]:
"Check whether the diffraction pattern of individual domain has the diffraction peaks of interest"
r_domain = [32, 160] #The index of a single domain, the [32,160] is the index for a domain in the example dataset
dp_single_domain = bindata[r_domain[0], r_domain[1]] #get the diffraction pattern of the single domain
im, contrast = plot_slider(dp_single_domain, title='Diffraction Pattern of a Single Domain') #display the diffraction pattern of the single domain with a slider to change the contrast and gamma
noframeimshow(im, contrast=contrast, greyscale=True) #display the diffraction pattern of the single domain without frame for saving


Isolate Homogeneous Signals

In [17]:
"""First create a annular virtual aperture to include most fluctuating diffuse scattering signals
The parameters given below are for the example dataset, should be modified for different datasets"""
inner_radius = 20 #inner radius of the annular virtual aperture
outer_radius = 40 #outer radius of the annular virtual aperture
center = [74, 75]
vapt_annular = np.zeros_like(I_diff[0,0]).astype(np.uint8) #initialize the annular virtual aperture, set datatype to uint8 for cv2.circle
vapt_annular = cv2.circle(vapt_annular, tuple([center[1], center[0]]), outer_radius, 1, -1) #Fill the outer circle with value 1
vapt_annular = cv2.circle(vapt_annular, tuple([center[1], center[0]]), inner_radius, 0, -1) #Fill the inner circle with value 0 to create the annular virtual aperture

"visualize the annular virtual aperture with the normalized variance map, check if the parameters are correct"
plt.figure()
plt.imshow(V)
plt.imshow(vapt_annular, alpha=0.2)
plt.show()


In [18]:
all_domains_map = virtual_dark_field(I_diff, vapt_annular) #calculate the virtual dark field of all the fluctuating diffuse scattering signals selected by the annular virtual aperture
noframeimshow(all_domains_map) #display the virtual dark field of all the fluctuating diffuse scattering signals without frame for saving
homogeneous_roi = np.zeros_like(all_domains_map).astype(int) #initialize the region of interest for homogeneous diffuse scattering
homogeneous_roi[all_domains_map<=np.percentile(all_domains_map, 5)] = 1 #set the pixels where the virtual dark field value below its 5th percentile to 1
homogeneous_dp = SADP(bindata, homogeneous_roi) #calculate the averaged diffraction pattern of the homogeneous diffuse scattering
im, contrast = plot_slider(homogeneous_dp, title='Averaged Diffraction Pattern of Homogeneous Diffuse Scattering') #display the averaged diffraction pattern of the homogeneous diffuse scattering with a slider to change the contrast and gamma
noframeimshow(im, contrast=contrast, greyscale=True) #display the averaged diffraction pattern of the homogeneous diffuse scattering without frame for saving
